# 09 RAG Evaluation Frameworks & Metrics (RAGAS)

**Real-World Scenario**: Enterprise RAG QA Benchmarking Suite.

This notebook demonstrates **Mean Reciprocal Rank (MRR)** and **NDCG@K** IR metric evaluation calculations, completed by an **End-to-End LLM-as-a-Judge Faithfulness Call**.

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from repository root .env
load_dotenv()

# Create hidden vector database directory for local index persistence
os.makedirs(".vectordb", exist_ok=True)

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden DB Directory Path:", os.path.abspath(".vectordb"))


## Step 2: IR Metric Calculations (MRR & NDCG@K)

In [ ]:
import numpy as np

def compute_mrr(rank_positions: list[int]) -> float:
    return float(np.mean([1.0 / r for r in rank_positions]))

def compute_ndcg_at_k(relevance_scores: list[int], k: int = 3) -> float:
    dcg = sum([(2**rel - 1) / np.log2(idx + 2) for idx, rel in enumerate(relevance_scores[:k])])
    ideal = sorted(relevance_scores, reverse=True)
    idcg = sum([(2**rel - 1) / np.log2(idx + 2) for idx, rel in enumerate(ideal[:k])])
    return float(dcg / idcg) if idcg > 0 else 0.0

ranks = [1, 2, 1, 4]
relevance = [3, 0, 2, 1]

print("=== Information Retrieval Evaluation Benchmark ===")
print(f"Mean Reciprocal Rank (MRR): {compute_mrr(ranks):.4f}")
print(f"NDCG@3 Score:              {compute_ndcg_at_k(relevance, k=3):.4f}")


## Step 3: Executing Complete LLM-as-a-Judge Faithfulness Evaluation Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

if os.getenv("OPENAI_API_KEY"):
    judge_prompt = ChatPromptTemplate.from_template('''You are an Evaluator Judge LLM.
Evaluate if the Generated Answer is fully grounded in the Context below.
Output 'FAITHFUL: TRUE' or 'FAITHFUL: FALSE' with 1 sentence rationale.

Context: {context}
Generated Answer: {answer}
Evaluation:''')
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    context_sample = "Microservice A communicates with Database B via gRPC over TLS 1.3."
    answer_sample = "Microservice A uses gRPC over TLS 1.3."
    
    response = llm.invoke(judge_prompt.format(context=context_sample, answer=answer_sample))
    print("\n=== Complete LLM-as-a-Judge Evaluation Response ===")
    print(response.content)
